# 01 – Exploratory Data Analysis (EDA)

**Goal:** understand the data *before* building any model.

Questions to answer:
1. How big is the data? What does each column mean?
2. What problems does it have (missing values, duplicates, typos, outliers)?
3. How is the target distributed? Is it imbalanced?
4. Which features seem related to the target?

> 👉 Write your findings in the **Key findings** cell at the end. They go into your report.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (notebooks/ is one level down)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml import config

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Load the raw data

In [ ]:
df = pd.read_csv(config.RAW_DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 2. Data quality checks

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
    "unique": df.nunique(),
})
print("Duplicate rows:", df.duplicated().sum())
quality

Text columns often hide problems. Look at the spellings 👇

In [ ]:
for col in ["province", "payment_method", "churn"]:
    print(df[col].value_counts(dropna=False), "\n")

In [ ]:
# Numbers stored as text? (e.g. ' ' in total_charges_usd)
bad = pd.to_numeric(df["total_charges_usd"], errors="coerce").isna() & df["total_charges_usd"].notna()
print("Non-numeric total_charges_usd values:", bad.sum(), df.loc[bad, "total_charges_usd"].unique())

# Impossible values?
age = pd.to_numeric(df["age"], errors="coerce")
print("Impossible ages:", sorted(age[(age < 16) | (age > 100)].unique()))

In [ ]:
# Outliers – look at the box plots
num_cols = ["age", "tenure_months", "monthly_charges_usd", "data_usage_gb", "support_calls", "late_payments"]
tmp = df[num_cols].apply(pd.to_numeric, errors="coerce")
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(x=tmp[col], ax=ax)
plt.tight_layout(); plt.show()

## 3. Target variable

For EDA we quickly normalise the labels (proper cleaning is in notebook 02).

In [ ]:
target = df[config.TARGET_COLUMN].str.strip().str.lower().map({"yes": "Yes", "y": "Yes", "no": "No", "n": "No"})
print(target.value_counts(normalize=True).round(3))
target.value_counts().plot.bar(title="Target distribution", color=["#4472C4", "#ED7D31"])
plt.show()

## 4. Features vs target

In [ ]:
eda = df.assign(churn=target, age=tmp["age"])
cat_cols = ["contract_type", "internet_service", "has_mobile_app", "gender"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, cat_cols):
    (eda.groupby(col)["churn"].apply(lambda s: (s == "Yes").mean())
        .sort_values().plot.barh(ax=ax, color="#4472C4"))
    ax.set_title(f"Churn rate by {col}"); ax.set_xlabel("churn rate")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["tenure_months", "monthly_charges_usd", "support_calls", "late_payments"]):
    sns.boxplot(data=eda[eda["monthly_charges_usd"] < 200], x="churn", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
corr = eda[num_cols].assign(churn=(eda["churn"] == "Yes").astype(int)).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix"); plt.show()

## 5. Key findings ✍️

👉 TODO: replace with YOUR findings. Example:

- 3,035 rows × 16 columns; 35 duplicate rows.
- Problems: missing `age` and `internet_service`, impossible ages (0, 150, 999),
  blank `total_charges_usd`, inconsistent spellings in `province`, `payment_method`, `churn`,
  extreme `monthly_charges_usd` outliers.
- Target: ~39 % churn → moderately imbalanced; accuracy alone is not enough, use F1 / recall.
- Month-to-month contracts, many support calls and late payments → higher churn.
- Longer tenure and using the mobile app → lower churn.